## Step 1

In [ ]:
#!pip install pandas openpyxl
#!pip install cvzone mediapipe

#### Load Excel File

In [1]:
import pandas as pd

file_path = r"D:\Innomatics\6) ML\5) Projects\1) OpenCV\gesture_task_dashboard_tasks.xlsx"

df = pd.read_excel(file_path)

df

,No,Title,Task,IMP
0,1,Workout Sprint,Complete a 30-minute high-intensity workout se...,2
1,2,Portfolio Update,Update GitHub with latest project and improve ...,3
2,3,Read & Reflect,Read 20 pages of a self-development book and w...,1
3,4,Skill Upgrade,Practice 10 coding problems on data structures,3
4,5,Call Parents,Have a meaningful 15-minute conversation with ...,2
5,6,Budget Review,Analyze monthly expenses and optimize unnecess...,2
6,7,Creative Break,Design a small poster or graphic using Canva o...,1


In [2]:
print(df.head())

   No             Title                                               Task  \
0   1    Workout Sprint  Complete a 30-minute high-intensity workout se...   
1   2  Portfolio Update  Update GitHub with latest project and improve ...   
2   3    Read & Reflect  Read 20 pages of a self-development book and w...   
3   4     Skill Upgrade     Practice 10 coding problems on data structures   
4   5      Call Parents  Have a meaningful 15-minute conversation with ...   

   IMP  
0    2  
1    3  
2    1  
3    3  
4    2  


In [3]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   No      7 non-null      int64
 1   Title   7 non-null      str  
 2   Task    7 non-null      str  
 3   IMP     7 non-null      int64
dtypes: int64(2), str(2)
memory usage: 356.0 bytes
None


#### Create Task Class and Convert DataFrame to Task Objects

In [4]:
class Task:
    def __init__(self, no, title, description, importance):
        self.id = no
        self.title = title
        self.description = description
        self.importance = importance
        self.is_completed = False
        self.position = [0, 0]

In [5]:
tasks = []

for _, row in df.iterrows():
    task = Task(
        no=row["No"],
        title=row["Title"],
        description=row["Task"],
        importance=row["IMP"]
    )
    tasks.append(task)

print(f"Total Tasks Loaded: {len(tasks)}")

Total Tasks Loaded: 7


## Step 2

#### Decide Screen Layout

#### Assign Initial Positions

In [6]:
# Screen dimensions
SCREEN_WIDTH = 1280
SCREEN_HEIGHT = 720

# Task card size
CARD_WIDTH = 220
CARD_HEIGHT = 120

# Starting position
start_x = 100
start_y = 150

gap_x = 250
gap_y = 160

for i, task in enumerate(tasks):
    row = i // 3
    col = i % 3
    
    x = start_x + col * gap_x
    y = start_y + row * gap_y
    
    task.position = [x, y]

#### Tutorial

In [7]:
import cv2
from cvzone.HandTrackingModule import HandDetector

# -----------------------------
# Camera Setup
# -----------------------------

cap = cv2.VideoCapture(0)
cap.set(3,1280)
cap.set(4,720)

detector = HandDetector(detectionCon=0.7, maxHands=1)

# -----------------------------
# Load Tutorial Images
# -----------------------------

img1 = cv2.imread(r"D:\Innomatics\6) ML\5) Projects\1) OpenCV\Presentation\1) Pics\1.png")
img2 = cv2.imread(r"D:\Innomatics\6) ML\5) Projects\1) OpenCV\Presentation\1) Pics\2.png")

img1 = cv2.resize(img1,(1280,720))
img2 = cv2.resize(img2,(1280,720))

state = "IMAGE1"

# -----------------------------
# Text Background Helper
# -----------------------------

def draw_text_with_bg(img, text, pos, scale=0.8, thickness=2):
    (w, h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thickness)
    x, y = pos

    # background rectangle
    cv2.rectangle(img,
                  (x-20, y-h-20),
                  (x+w+20, y+10),
                  (40,40,40),
                  -1)

    # text
    cv2.putText(img,
                text,
                (x,y),
                cv2.FONT_HERSHEY_SIMPLEX,
                scale,
                (255,255,255),
                thickness)

# -----------------------------
# Main Loop
# -----------------------------

while True:

    success, frame = cap.read()
    frame = cv2.flip(frame,1)

    hands, frame = detector.findHands(frame)

    pinch = False
    triple = False

    if hands:
        lm = hands[0]["lmList"]

        p_index = lm[8][:2]
        p_middle = lm[12][:2]
        p_thumb = lm[4][:2]

        dist_im,_,_ = detector.findDistance(p_index,p_middle,frame)
        dist_it,_,_ = detector.findDistance(p_index,p_thumb,frame)

        pinch = dist_im < 40
        triple = dist_im < 40 and dist_it < 40

    # -----------------------------------
    # IMAGE 1
    # -----------------------------------

    if state == "IMAGE1":

        screen = img1.copy()

        draw_text_with_bg(
            screen,
            "Press T to TRY gesture",
            (420,680),
            0.9
        )

    # -----------------------------------
    # TRY GESTURE 1
    # -----------------------------------

    elif state == "TRY1":

        screen = frame.copy()

        draw_text_with_bg(
            screen,
            "Perform Gesture: Touch INDEX + MIDDLE finger",
            (350,80),
            0.8
        )

        if pinch:
            draw_text_with_bg(
                screen,
                "Gesture Detected!",
                (520,150),
                1
            )

        draw_text_with_bg(
            screen,
            "Press V to view image | Press N for next",
            (350,680),
            0.8
        )

    # -----------------------------------
    # IMAGE 2
    # -----------------------------------

    elif state == "IMAGE2":

        screen = img2.copy()

        draw_text_with_bg(
            screen,
            "Press T to TRY gesture",
            (420,680),
            0.9
        )

    # -----------------------------------
    # TRY GESTURE 2
    # -----------------------------------

    elif state == "TRY2":

        screen = frame.copy()

        draw_text_with_bg(
            screen,
            "Perform Gesture: INDEX + MIDDLE + THUMB",
            (380,80),
            0.8
        )

        if triple:
            draw_text_with_bg(
                screen,
                "Triple Pinch Detected!",
                (460,150),
                1
            )

        draw_text_with_bg(
            screen,
            "Press B to go BACK | Press Q to QUIT",
            (350,680),
            0.8
        )

    # -----------------------------------

    cv2.imshow("Gesture Tutorial",screen)

    key = cv2.waitKey(1) & 0xFF

    # -----------------------------------
    # KEY CONTROLS
    # -----------------------------------

    if key == ord('t'):

        if state == "IMAGE1":
            state = "TRY1"

        elif state == "IMAGE2":
            state = "TRY2"

    elif key == ord('v') and state == "TRY1":
        state = "IMAGE1"

    elif key == ord('n') and state == "TRY1":
        state = "IMAGE2"

    elif key == ord('b') and state == "TRY2":
        state = "IMAGE1"

    elif key == ord('q'):
        break

    elif key == 27:  # ESC
        break


cap.release()
cv2.destroyAllWindows()

#### Render Static UI

In [20]:
## Final

In [9]:
import cv2
from cvzone.HandTrackingModule import HandDetector
import math
import time
import random

# --- Initialization ---
detector = HandDetector(detectionCon=0.6, maxHands=1)

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)
cap.set(cv2.CAP_PROP_BRIGHTNESS, 150)

SCREEN_WIDTH = 1280
SCREEN_HEIGHT = 720
COMPLETION_ZONE_WIDTH = 150
CARD_WIDTH = 220
CARD_HEIGHT = 120

selected_task = None
info_task = None
cursor = None
prev_cursor = None
smooth_factor = 0.4

filter_mode = 0
view_state = "START"  # New state: "START" or "DASHBOARD"

display_progress = 0
wave_offset = 0

# Positions for Start Menu (Middle)
start_buttons = {
    1: (200, 300, 200, 100),
    2: (540, 300, 200, 100),
    3: (880, 300, 200, 100)
}

# Positions for Dashboard (Top)
filter_buttons = {
    1: (200, 20, 200, 60),
    2: (500, 20, 200, 60),
    3: (800, 20, 200, 60)
}

celebration_mode = False
confetti_particles = []

while True:
    success, img = cap.read()
    if not success: break
    img = cv2.flip(img, 1)

    hands, img = detector.findHands(img)

    if hands:
        hand = hands[0]
        lmList = hand["lmList"]
        p_index = lmList[8][:2]
        p_middle = lmList[12][:2]
        p_thumb = lmList[4][:2]

        l_im, _, _ = detector.findDistance(p_index, p_middle, img)
        l_it, _, _ = detector.findDistance(p_index, p_thumb, img)

        if prev_cursor is None:
            cursor = p_index
        else:
            cursor = [
                int(prev_cursor[0] + (p_index[0] - prev_cursor[0]) * smooth_factor),
                int(prev_cursor[1] + (p_index[1] - prev_cursor[1]) * smooth_factor)
            ]
        prev_cursor = cursor

        triple_pinch = (l_im < 45 and l_it < 45)
        normal_pinch = (l_im < 45)

        if triple_pinch and view_state == "DASHBOARD":
            selected_task = None
            info_task = None
            for task in tasks:
                if task.is_completed: continue
                if task.importance != filter_mode: continue
                x, y = task.position
                if x < cursor[0] < x + CARD_WIDTH and y < cursor[1] < y + CARD_HEIGHT:
                    info_task = task
                    break

        elif normal_pinch:
            info_task = None
            # Decide which button set to check based on state
            current_buttons = start_buttons if view_state == "START" else filter_buttons
            for mode, (x, y, w, h) in current_buttons.items():
                if x < cursor[0] < x + w and y < cursor[1] < y + h:
                    filter_mode = mode
                    view_state = "DASHBOARD"
            
            if view_state == "DASHBOARD" and selected_task is None:
                for task in tasks:
                    if task.is_completed: continue
                    if task.importance != filter_mode: continue
                    x, y = task.position
                    if x < cursor[0] < x + CARD_WIDTH and y < cursor[1] < y + CARD_HEIGHT:
                        selected_task = task
                        break
        else:
            if selected_task:
                if cursor and cursor[0] > (SCREEN_WIDTH - COMPLETION_ZONE_WIDTH):
                    selected_task.is_completed = True
            selected_task = None

    if selected_task and cursor:
        selected_task.position = [cursor[0] - CARD_WIDTH // 2, cursor[1] - CARD_HEIGHT // 2]

        # ADD TITLE ONLY WHEN IN START SCREEN
    if view_state == "START":
        text = "TODAY'S TASK"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.4, 3)
        x, y = 480, 150
    
        # background rectangle
        cv2.rectangle(img, (x-20, y-th-20), (x+tw+20, y+10), (40,40,40), -1)
    
        # text
        cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255,255,255), 3)
    
    # 1. Draw Buttons
    active_buttons = start_buttons if view_state == "START" else filter_buttons
    for mode, (x, y, w, h) in active_buttons.items():
        color = (0, 200, 0) if mode == 1 else (0, 200, 255) if mode == 2 else (0, 0, 255)
        label = "NORMAL" if mode == 1 else "MEDIUM" if mode == 2 else "HIGH"
        # Grey background always
        cv2.rectangle(img, (x, y), (x + w, y + h), (80, 80, 80), -1)
        if filter_mode == mode:
            cv2.rectangle(img, (x, y), (x + w, y + h), color, -1)
            cv2.rectangle(img, (x, y), (x + w, y + h), (255,255,255), 2)
        else:
            cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
        
        # Center text in buttons
        font_scale = 1.0 if view_state == "START" else 0.7
        cv2.putText(img, label, (x + 20, y + h//2 + 10), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255,255,255), 2)

    if view_state == "DASHBOARD":
        # 2. Task Cards
        completed_count = 0
        total_tasks = len(tasks)
        for task in tasks:
            if task.is_completed:
                completed_count += 1
                continue
            if task.importance != filter_mode: continue
            
            x, y = task.position
            base_color = (0, 200, 0) if task.importance == 1 else (0, 200, 255) if task.importance == 2 else (0, 0, 255)
            badge = "N" if task.importance == 1 else "M" if task.importance == 2 else "H"
            cv2.rectangle(img, (x, y), (x + CARD_WIDTH, y + CARD_HEIGHT), base_color, -1)
            cv2.circle(img, (x + CARD_WIDTH - 20, y + 20), 15, (255,255,255), -1)
            cv2.putText(img, badge, (x + CARD_WIDTH - 28, y + 27), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2)
            cv2.putText(img, task.title, (x + 15, y + 65), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        # 3. Completion Zone
        zone_left = SCREEN_WIDTH - COMPLETION_ZONE_WIDTH
        hover_zone = cursor and cursor[0] > zone_left
        cv2.rectangle(img, (zone_left, 0), (SCREEN_WIDTH, SCREEN_HEIGHT), (30,30,30), -1)
        if hover_zone:
            glow = int(100 + 80 * abs(math.sin(time.time() * 4)))
            cv2.rectangle(img, (zone_left, 0), (SCREEN_WIDTH, SCREEN_HEIGHT), (0, glow, 0), 4)
        cv2.putText(img, "DROP", (zone_left + 30, SCREEN_HEIGHT // 2 - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        cv2.putText(img, "HERE", (zone_left + 30, SCREEN_HEIGHT // 2 + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)

        # 4. Progress Bar (Updated with Dynamic Color Logic)
        target_progress = int((completed_count / total_tasks) * 100) if total_tasks > 0 else 0
        display_progress += (target_progress - display_progress) * 0.1
        progress = int(display_progress)
        
        # --- Color Logic ---
        if progress < 35:
            bar_color = (0, 0, 255)    # Red for low progress
        elif progress < 75:
            bar_color = (0, 255, 255)  # Yellow/Orange for medium
        else:
            bar_color = (0, 255, 100)  # Green for high completion

        bar_x, bar_y, bar_w, bar_h = 200, SCREEN_HEIGHT - 60, 800, 30
        cv2.rectangle(img, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (60, 60, 60), 2)
        
        filled_width = int((progress / 100) * bar_w)
        wave_offset += 0.2
        for x in range(filled_width):
            wave = int(4 * math.sin((x * 0.05) + wave_offset))
            # Drawing the river flow with the dynamic bar_color
            cv2.line(img, (bar_x + x, bar_y + wave), (bar_x + x, bar_y + bar_h), bar_color, 1)
            
        cv2.putText(img, f"{progress}%", (bar_x + bar_w + 20, bar_y + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        # 5. Info Popup (Same logic)
        if info_task:
            overlay = img.copy()
            cv2.rectangle(overlay, (200,150), (1080,550), (40,40,40), -1)
            img = cv2.addWeighted(overlay, 0.9, img, 0.1, 0)
            cv2.rectangle(img, (200,150), (1080,550), (255,255,255), 2)
            cv2.putText(img, info_task.title, (250,220), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
            words = info_task.description.split()
            line, y_offset = "", 280
            for word in words:
                if len(line + word + " ") > 70:
                    cv2.putText(img, line, (250, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200,200,200), 2)
                    y_offset += 40
                    line = word + " "
                else: line += word + " "
            cv2.putText(img, line, (250, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200,200,200), 2)

    # 6. Celebration Screen
    if celebration_mode:
        overlay = img.copy()
        cv2.rectangle(overlay, (0,0), (SCREEN_WIDTH, SCREEN_HEIGHT), (0,0,0), -1)
        img = cv2.addWeighted(overlay, 0.8, img, 0.2, 0)
        bounce = int(20 * abs(math.sin(time.time() * 3)))
        text = "YAY! You Completed All Tasks :)"
        text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)[0]
        cv2.putText(img, text, ((SCREEN_WIDTH - text_size[0]) // 2, SCREEN_HEIGHT // 2 - bounce), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,255), 3)

        if len(confetti_particles) < 100:
            confetti_particles.append([random.randint(0, SCREEN_WIDTH), SCREEN_HEIGHT, 
                                      random.randint(0,255), random.randint(0,255), 
                                      random.randint(0,255), random.uniform(-2,2)])
        for p in confetti_particles:
            p[1] -= random.uniform(2,5); p[0] += p[5]
            cv2.circle(img, (int(p[0]), int(p[1])), 4, (p[2], p[3], p[4]), -1)
    elif 'completed_count' in locals() and 'total_tasks' in locals() and completed_count == total_tasks and total_tasks != 0:
        celebration_mode = True

    cv2.imshow("Gesture Task Dashboard", img)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()